[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_26/VL26_BERT_Embeddings.ipynb)

# VL26 Notebook 2: BERT und kontextabhängige Wortvektoren

**BERT** steht für *Bidirectional Encoder Representations from Transformers*. Es ist ein Sprachmodell, das Text mithilfe eines neuronalen Netzes verarbeitet. „Bidirectional“ bedeutet hier: Für die Darstellung eines Wortes berücksichtigt BERT Wörter links und rechts davon.

Ein **Embedding** ist ein Vektor, also eine Liste von Zahlen. Anders als bei Word2Vec kann der Vektor einer Wortform bei BERT je nach Satz unterschiedlich ausfallen. Wir untersuchen das am Beispiel „Bank“ und probieren anschließend ein separates Modell für Sentiment-Analyse aus.

Wir verwenden vortrainierte Modelle von Hugging Face; ein API-Key ist nicht nötig. Beim ersten Laden ist eine Internetverbindung erforderlich. Die Berechnung läuft in der Notebook-Umgebung, etwa lokal oder in Colab.

In [ ]:
# Installation (nur einmal ausführen)
# !pip install transformers torch numpy scikit-learn

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

## 1. BERT-Modell laden

Wir laden ein bereits trainiertes deutsches BERT-Modell und den zugehörigen **Tokenizer**. Der Tokenizer zerlegt Text in Token: Das können ganze Wörter, Wortteile oder Satzzeichen sein. BERT verarbeitet diese Token und berechnet ihre Vektoren. Wir trainieren das Modell hier nicht selbst.

In [ ]:
# Deutsches BERT-Modell laden
model_name = "bert-base-german-cased"
print(f"Lade Modell: {model_name}")

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

print("✓ Modell geladen!")

## 2. Das "Bank"-Beispiel: Zwei Kontexte, zwei Vektoren

Im Word2Vec-Notebook hat die Wortform „Bank“ einen festen Vektor. BERT berechnet dagegen für jedes Vorkommen eine Darstellung anhand des Satzes.

Wir vergleichen zwei Sätze: In einem geht es um ein Geldinstitut, im anderen um ein Sitzmöbel. Die Funktion unten sucht das Wort im tokenisierten Satz und entnimmt seinen Vektor aus der letzten Modellschicht. Besteht das Wort aus mehreren Token, mitteln wir deren Vektoren.

Die **Kosinus-Ähnlichkeit** vergleicht die Richtung der beiden Vektoren. Ein Wert nahe 1 bedeutet ähnliche Richtungen; er ist keine Wahrscheinlichkeit und kein direkter Beweis dafür, dass BERT die Bedeutungen richtig unterscheidet.

In [ ]:
# Zwei Sätze mit unterschiedlicher Bedeutung von "Bank"
satz1 = "Ich gehe zur Bank und hole Geld."
satz2 = "Ich sitze auf der Bank im Park."

def get_word_embedding(sentence, word, tokenizer, model):
    """Extrahiert den Vektor eines Wortes; Teilwort-Vektoren werden gemittelt."""
    # Tokenisierung
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.tokenize(sentence)
    
    # BERT durchlaufen (ohne Gradienten)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Letzter Hidden State: [batch_size, seq_len, hidden_size]
    last_hidden = outputs.last_hidden_state[0]  # [seq_len, 768]
    
    # Finde die Tokenfolge des Wortes im Satz
    word_tokens = tokenizer.tokenize(word)
    for i in range(len(tokens) - len(word_tokens) + 1):
        if tokens[i:i + len(word_tokens)] == word_tokens:
            # +1 wegen [CLS]; mehrere Teilwörter zu einem Vektor mitteln
            return last_hidden[i + 1:i + 1 + len(word_tokens)].mean(dim=0).numpy()
    
    raise ValueError(f"Wort '{word}' wurde im Satz nicht gefunden.")

# Embeddings für "Bank" in beiden Kontexten
vec1 = get_word_embedding(satz1, "Bank", tokenizer, model)
vec2 = get_word_embedding(satz2, "Bank", tokenizer, model)

# Ähnlichkeit berechnen
similarity = cosine_similarity([vec1], [vec2])[0][0]

print("Satz 1:", satz1)
print("Satz 2:", satz2)
print("Token für 'Bank':", tokenizer.tokenize("Bank"))
print("Anzahl der Werte pro Vektor:", len(vec1))
print(f"Kosinus-Ähnlichkeit der 'Bank'-Vektoren: {similarity:.4f}")

### Vergleich: gleiche und unterschiedliche Bedeutung

Wir ergänzen einen dritten Satz, in dem „Bank“ wie in Satz 1 ein Geldinstitut bezeichnet. Dann vergleichen wir den Vektor aus Satz 1 mit den Vektoren aus Satz 2 und Satz 3.

Ist die Ähnlichkeit bei gleicher Bedeutung höher? Auch wenn das für diese Sätze zutrifft, ist es kein allgemeiner Beweis: Die Vektoren hängen vom ganzen Satz ab, also auch von Wortwahl und Satzbau.

In [ ]:
satz3 = "Sie zahlt bei der Bank Geld ein."
vec3 = get_word_embedding(satz3, "Bank", tokenizer, model)

gleiche_bedeutung = cosine_similarity([vec1], [vec3])[0][0]
unterschiedliche_bedeutung = cosine_similarity([vec1], [vec2])[0][0]

print("Satz 1 (Geldinstitut):", satz1)
print("Satz 2 (Sitzmöbel):   ", satz2)
print("Satz 3 (Geldinstitut):", satz3)
print(f"\nSatz 1 und 3 – gleiche Bedeutung:         {gleiche_bedeutung:.4f}")
print(f"Satz 1 und 2 – unterschiedliche Bedeutung: {unterschiedliche_bedeutung:.4f}")

## 3. Weitere Wörter im Vergleich

Ein fester Word2Vec-Vektor für dieselbe Wortform hat mit sich selbst die Kosinus-Ähnlichkeit 1. Bei BERT vergleichen wir dagegen die jeweils im Satz berechneten Vektoren. Auch ähnliche Sätze können unterschiedliche Vektoren ergeben – nicht nur Sätze mit verschiedenen Wortbedeutungen.

Die folgenden Beispiele vergleichen „Schloss“ und „Ball“ in je zwei Sätzen. Ein einzelner Ähnlichkeitswert reicht nicht aus, um die Qualität des Modells zu beurteilen.

In [ ]:
# Weitere Beispiele mit mehrdeutigen Wörtern
beispiele = [
    ("Das Schloss ist sehr alt.", "Ich öffne das Schloss mit einem Schlüssel.", "Schloss"),
    ("Der Ball ist rund.", "Wir gehen zum Ball und tanzen.", "Ball"),
]

for satz_a, satz_b, wort in beispiele:
    vec_a = get_word_embedding(satz_a, wort, tokenizer, model)
    vec_b = get_word_embedding(satz_b, wort, tokenizer, model)
    sim = cosine_similarity([vec_a], [vec_b])[0][0]
    print(f"\n{wort}:")
    print(f"  A: {satz_a}")
    print(f"  B: {satz_b}")
    print(f"  Kosinus-Ähnlichkeit: {sim:.4f}")

## 4. Sentiment-Analyse mit BERT (Bonus)

**Sentiment-Analyse** ordnet einem Text eine Stimmung zu, zum Beispiel positiv oder negativ. Dafür laden wir ein zweites Modell: Es basiert auf DistilBERT und wurde zusätzlich für die Einordnung englischer Texte trainiert. Das deutsche BERT-Modell von oben wird hier nicht weitertrainiert oder verwendet.

Dieses Modell kennt nur die Klassen „positiv“ und „negativ“. Auch ein neutraler Satz muss daher einer dieser Klassen zugeordnet werden. Der ausgegebene Wert ist die Modellbewertung für die gewählte Klasse, keine Garantie für eine korrekte Antwort.

In [ ]:
from transformers import pipeline

# Separates englisches Modell für zwei Stimmungsklassen
print("Lade Sentiment-Modell (Englisch)...")
sentiment = pipeline("sentiment-analysis", 
                     model="distilbert-base-uncased-finetuned-sst-2-english",
                     framework="pt")  # Explizit PyTorch verwenden

# Test-Sätze (Englisch)
sätze = [
    "This product is fantastic!",
    "I am very disappointed with the quality.",
    "It is okay, nothing special.",
]

print("✓ Modell geladen!\n")
print("Sentiment-Analyse:\n")
for satz in sätze:
    result = sentiment(satz)[0]
    print(f"Satz: {satz}")
    print(f"  Vorhergesagte Klasse: {result['label']}")
    print(f"  Modellwert für diese Klasse: {result['score']:.2%}")
    print()

## 🎯 Takeaway

- **Word2Vec:** Ein fester Vektor pro Wortform.
- **BERT:** Ein Vektor pro Token-Vorkommen, der vom Kontext abhängt.
- **Vortraining:** Das Modell hat vor diesem Notebook aus vielen Texten Muster gelernt. Das garantiert keine fehlerfreien Ergebnisse.
- **Sentiment:** Für die Klassifikation haben wir ein anderes, dafür angepasstes Modell genutzt.

BERT verwendet eine Transformer-Architektur mit **Self-Attention**. Dabei können Token Informationen aus anderen Token des eingegebenen Textes einbeziehen.